# 数据集类

前面几章我们一直用一个样本（温度 28.1°C，湿度 58%，销量 165）训练模型。真实的训练需要大量样本反复迭代，才能让模型逐渐收敛。

本章引入**数据集**（Dataset）类，负责管理训练数据：把数据分为**训练集**和**测试集**，支持按批次读取（**批处理**），让训练循环可以自动遍历全部数据。

---

## 批处理

**批处理**（Mini-batch）是指每次训练不是用一个样本，而是用一小批样本同时计算梯度，再统一更新一次参数。批处理有两个好处：

* **效率**：矩阵运算可以并行，一次处理多个样本比逐个处理快得多；
* **稳定性**：多个样本的梯度取平均，比单个样本的梯度噪声更小，训练更稳定。

批处理会改变前向传播中张量的形状：单样本时特征是形如 `(2,)` 的一维向量，批处理时变成形如 `(batch_size, 2)` 的二维矩阵。这要求我们对线性层的梯度公式做相应调整。

In [1]:
from abc import ABC, abstractmethod

import numpy as np

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for parent in self.parents:
            parent.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

**基础数据集**（Dataset）是一个抽象类，封装了数据加载和迭代读取的逻辑。初始化时，它调用子类实现的加载函数 `load()` ，然后默认进入训练模式。

**模式切换函数**：
* `train()`：切换到训练模式，指向训练集；
* `eval()`：切换到测试模式，指向测试集。

**数据读取函数**：
* `items()`：返回当前模式下的全部数据；
* `__getitem__()`：按批次索引读取特征和标签张量，供训练循环逐批迭代；
* `__len__`：返回当前模式下的批次总数（样本数 ÷ 批大小）。

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

优化器添加一个**清零函数 `reset()`**，在每轮训练开始前将所有参数的梯度清零。由于张量的梯度 `grad` 在反向传播时累加，如果不清零，上一轮的梯度会叠加到下一轮，导致参数更新越来越大，偏离预期。这是深度学习训练中容易遗漏的关键步骤。

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

## 数据

### 冰激凌销量数据集

`IcecreamDataset` 加载了我们在第一部分使用的小明冰激凌店数据。
训练集包含 4 条历史记录，测试集包含 1 条用于最终评估。

| 集合 | 温度 (°C) | 湿度 (%) | 销量（支）|
|------|---------|--------|--------|
| 训练 | 22.5 | 72.0 | 95 |
| 训练 | 31.4 | 45.0 | 210 |
| 训练 | 19.8 | 85.0 | 70 |
| 训练 | 27.6 | 63.0 | 155 |
| **测试** | **28.1** | **58.0** | **165** |

In [7]:
class IcecreamDataset(Dataset):

    def load(self):
        self.train_features = [[22.5, 72.0],
                               [31.4, 45.0],
                               [19.8, 85.0],
                               [27.6, 63.0]]
        self.train_labels = [[95],
                             [210],
                             [70],
                             [155]]

        self.test_features = [[28.1, 58.0]]
        self.test_labels = [[165]]

## 模型

### 线性层

批处理改变了权重梯度的计算方式。

单样本时，输入 `x` 是形如 `(n_features,)` 的一维向量，梯度公式为：

$$
\frac{dL}{dW} = \delta \cdot x
$$

批处理时，`x` 变成形如 `(batch, n_features)` 的矩阵，`p.grad` 变成形如 `(batch, out)` 的矩阵。为了把批次中所有样本的梯度正确累加，公式变为矩阵乘法：

$$
\frac{dL}{dW} = \delta^T \cdot X
$$

维度验证：`p_grad.T` 形为 `(out, batch)`，`x` 形为 `(batch, n_features)`，乘积为 `(out, n_features)`，正好与 `weight.grad` 的形状 `(out, n_features)` 一致。

同时，对于偏置的计算方式，也要改成在批次维度上求和。

In [8]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.data.shape}; bias{self.bias.data.shape}]'

### 均方误差损失函数

批处理时，`y` 和 `p` 都是 `(batch, 1)` 的矩阵。
`np.mean()` 已经在计算损失时对整个批次取了平均，
但梯度 `dL/dp` 还需要除以批大小 `n`，才能与损失的平均语义保持一致：

$$
\frac{dL}{dp_i} = \frac{-2(y_i - p_i)}{n}
$$

这里用 `len(y.data)` 获取批大小（即 `y` 的第一个维度，样本数）。

In [9]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / len(y.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [10]:
class SGDOptimizer(Optimizer):

    def step(self):
        for param in self.parameters:
            param.data -= param.grad * self.lr

## 设置

### 学习率

In [11]:
LEARNING_RATE = 0.00001

### 批大小

批大小（`BATCH_SIZE`）决定每次训练用几个样本。
这里设为 `2`：4 条训练数据分成 2 批，每批 2 个样本。

In [12]:
BATCH_SIZE = 2

## 训练

### 建模

In [13]:
dataset = IcecreamDataset(BATCH_SIZE)

layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)
print(layer)

Linear[weight(1, 2); bias(1,)]


### 迭代

现在可以用数据集进行批次迭代训练。

`len(dataset)` 返回批次总数（4 样本 ÷ 批大小 2 = 2 批），所以这个循环会遍历全部训练数据各一次。

In [14]:
for i in range(len(dataset)):
    features, labels = dataset[i]

    # 1. 梯度清零
    optimizer.reset()
    # 2. 前向传播
    predictions = layer(features)
    # 3.计算损失
    loss = loss_fn(predictions, labels)
    # 4. 梯度计算（自动微分）
    loss.backward()
    # 5. 更新参数
    optimizer.step()

print(f'weight: {layer.weight}')
print(f'bias:   {layer.bias}')

batch 0: loss=Tensor(15897.651250000003)
batch 1: loss=Tensor(5111.254367594466)
weight: Tensor([[0.59388172 0.68104165]])
bias:   Tensor([0.00327249])


## 验证

### 推理

训练完成后，切换到测试模式，用测试集评估模型。`dataset.eval()` 把数据集重新指向测试集；`dataset.items()` 一次性返回测试集的全部数据（不按批次），直接用于推理。

In [15]:
dataset.eval()
features, labels = dataset.items()
predictions = layer(features)
print(f'prediction: {predictions}')
print(f'label:      {labels}')

prediction: Tensor([[56.19176426]])
label:      Tensor([[165]])


### 评估

In [16]:
loss = loss_fn(predictions, labels)
print(f'loss: {loss}')

loss: Tensor(11839.232164432306)


测试损失约为 `11839`，比初始的 `14871` 有所下降，说明训练是有效的。

## 课后练习

去掉 `optimizer.reset()` 后多轮训练会有什么问题？